In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [3]:
# Time
start = dt.datetime(2019,4,5)
end = dt.datetime(2019,5,25)
print(start,end,end-start)

2019-04-05 00:00:00 2019-05-25 00:00:00 50 days, 0:00:00


In [4]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$lt': end, '$gte': start}},{"sign_up_details":1, "created_at":1,"login_details":1}):
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
df_users = pd.DataFrame(dic_flattened)
df_users = df_users[df_users["sign_up_details_app_platform"] == "UNITY_Android"]
users = df_users[["_id","created_at","sign_up_details_device_id","login_details_last_request_at"]]
users.columns = ["user_id","createtime","device_id","last_request"]
users.head()

,user_id,createtime,device_id,last_request
0,5ca6adb3b65b15544e169963,2019-04-05 01:21:55.931,ba643254ecf37ea8a8453a12ab14e7a8,2019-04-05 01:22:16.059
1,5ca6c1bc8c899454486ac253,2019-04-05 02:47:24.829,3ac01ce3f63c2c857064702d99c80541,2019-04-05 03:09:53.215
2,5ca6d01fd468e03534a80f6d,2019-04-05 03:48:47.021,2534dd971bd5601f2985c94d1da74cc9,2019-04-05 03:51:56.000
4,5ca6e3fc800ebb353a469cd1,2019-04-05 05:13:32.630,bdfe19ca2338a8176670c5c79695b36f,2019-04-05 10:11:34.754
5,5ca6e9a3acb9c30617a1e75f,2019-04-05 05:37:39.265,a395477c3bd6830b7e0c68571da67ed8,2019-04-05 05:37:44.384


In [5]:
users.sort_values(['device_id','createtime'],inplace = True)
users = users.drop_duplicates('device_id')

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.


In [6]:
users = users[users['last_request']-users['createtime']>'72:00:00']

In [7]:
team_cursor = cursor.superstars.teams
aw_teams = []
for documents in team_cursor.find({'created_at': {'$lt': end, '$gte': start}},{"user":1, "created_at":1}):
    aw_teams.append(documents)
dic_flattened = [flatten(d) for d in aw_teams]
df_teams = pd.DataFrame(dic_flattened)
teams = df_teams[df_teams["user"].isin(users["user_id"])]
teams = teams[["_id","user","created_at"]]
teams.columns = ["team_id", "user_id", "team_created_at"]
teams.head()

,team_id,user_id,team_created_at
21,5ca6ead6b65b15544e169afb,5ca6ead5b65b15544e169ad6,2019-04-05 05:42:46.072
22,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662
43,5ca6ec2c93aec35428abbb1e,5ca6ec2c93aec35428abbaf9,2019-04-05 05:48:28.567
54,5ca6ed54d468e03534a811f7,5ca6ed54d468e03534a811d2,2019-04-05 05:53:24.507
79,5ca6eea3800ebb353a46a000,5ca6eea3800ebb353a469fdb,2019-04-05 05:58:59.458


In [8]:
users_team = pd.merge(teams,users,on='user_id')

In [9]:
users_team.head()

,team_id,user_id,team_created_at,createtime,device_id,last_request
0,5ca6ead6b65b15544e169afb,5ca6ead5b65b15544e169ad6,2019-04-05 05:42:46.072,2019-04-05 05:42:46.000,ba49f508d110ab8ebfc1e6cfdab3e476,2019-04-13 15:26:21.403
1,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662,2019-04-05 05:42:46.569,a34697c2e71c790e3ff1b58183aa4bc6,2019-04-10 17:10:31.047
2,5ca6ec2c93aec35428abbb1e,5ca6ec2c93aec35428abbaf9,2019-04-05 05:48:28.567,2019-04-05 05:48:28.484,ef6753e87c4fdf13ba0b5b62590f13f0,2019-04-08 11:09:02.042
3,5ca6ed54d468e03534a811f7,5ca6ed54d468e03534a811d2,2019-04-05 05:53:24.507,2019-04-05 05:53:24.410,3c6ba7bf8141ea0126bcdcd4d5892737,2019-05-07 10:15:42.682
4,5ca6eea3800ebb353a46a000,5ca6eea3800ebb353a469fdb,2019-04-05 05:58:59.458,2019-04-05 05:58:59.360,0f331606133063e1b259def52ffbccb3,2019-04-26 06:40:56.932


In [10]:
con_cursor = cursor.superstars.matches
aw_matches = []
for documents in con_cursor.find({'created_at': {'$lt': end, '$gte': start},"status":3}):
    aw_matches.append(documents)
dic_flattened = [flatten(d) for d in aw_matches]
df_matches = pd.DataFrame(dic_flattened)
df_matches = df_matches.rename(columns={'home_team_id':'team_id'})
matches = df_matches[df_matches["team_id"].isin(teams["team_id"])]
matches = matches.loc[:,["team_id","winner_team_id","type","start_time","home_team_total"]]

In [11]:
matches.head()

,team_id,winner_team_id,type,start_time,home_team_total
154,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 05:43:03.216,53.0
155,5ca6ead6b65b15544e169afb,5ca6ead6b65b15544e169afb,CAMPAIGN,2019-04-05 05:43:06.613,58.0
168,5ca6ead6b65b15544e169afb,5ca6ead6b65b15544e169afb,CAMPAIGN,2019-04-05 05:46:43.382,84.0
182,5ca6ec2c93aec35428abbb1e,5ca6ec2c93aec35428abbb1e,CAMPAIGN,2019-04-05 05:48:50.295,53.0
203,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 05:53:37.209,82.0


In [12]:
users_match = pd.merge(matches,users_team,on='team_id')

In [13]:
users_match.head()

,team_id,winner_team_id,type,start_time,home_team_total,user_id,team_created_at,createtime,device_id,last_request
0,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 05:43:03.216,53.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662,2019-04-05 05:42:46.569,a34697c2e71c790e3ff1b58183aa4bc6,2019-04-10 17:10:31.047
1,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 05:53:37.209,82.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662,2019-04-05 05:42:46.569,a34697c2e71c790e3ff1b58183aa4bc6,2019-04-10 17:10:31.047
2,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 05:58:01.033,66.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662,2019-04-05 05:42:46.569,a34697c2e71c790e3ff1b58183aa4bc6,2019-04-10 17:10:31.047
3,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 06:07:05.673,67.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662,2019-04-05 05:42:46.569,a34697c2e71c790e3ff1b58183aa4bc6,2019-04-10 17:10:31.047
4,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 07:29:20.018,94.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662,2019-04-05 05:42:46.569,a34697c2e71c790e3ff1b58183aa4bc6,2019-04-10 17:10:31.047


In [14]:
users_match.type.unique()

array(['CAMPAIGN', 'IPL', 'ENTRY_LEAGUE', 'TGPL'], dtype=object)

In [15]:
users_match = users_match[users_match['type']!='ENTRY_LEAGUE']

In [16]:
users_match.head()

,team_id,winner_team_id,type,start_time,home_team_total,user_id,team_created_at,createtime,device_id,last_request
0,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 05:43:03.216,53.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662,2019-04-05 05:42:46.569,a34697c2e71c790e3ff1b58183aa4bc6,2019-04-10 17:10:31.047
1,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 05:53:37.209,82.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662,2019-04-05 05:42:46.569,a34697c2e71c790e3ff1b58183aa4bc6,2019-04-10 17:10:31.047
2,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 05:58:01.033,66.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662,2019-04-05 05:42:46.569,a34697c2e71c790e3ff1b58183aa4bc6,2019-04-10 17:10:31.047
3,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 06:07:05.673,67.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662,2019-04-05 05:42:46.569,a34697c2e71c790e3ff1b58183aa4bc6,2019-04-10 17:10:31.047
4,5ca6ead6d468e03534a810ee,5ca6ead6d468e03534a810ee,CAMPAIGN,2019-04-05 07:29:20.018,94.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.662,2019-04-05 05:42:46.569,a34697c2e71c790e3ff1b58183aa4bc6,2019-04-10 17:10:31.047


In [17]:
users_match.drop(['winner_team_id','type','team_created_at','device_id','last_request'],axis=1,inplace=True)

In [18]:
users_match = users_match[users_match['start_time']-users_match['createtime']<'72:00:00']

In [19]:
users_match.head()

,team_id,start_time,home_team_total,user_id,createtime
0,5ca6ead6d468e03534a810ee,2019-04-05 05:43:03.216,53.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.569
1,5ca6ead6d468e03534a810ee,2019-04-05 05:53:37.209,82.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.569
2,5ca6ead6d468e03534a810ee,2019-04-05 05:58:01.033,66.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.569
3,5ca6ead6d468e03534a810ee,2019-04-05 06:07:05.673,67.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.569
4,5ca6ead6d468e03534a810ee,2019-04-05 07:29:20.018,94.0,5ca6ead6d468e03534a810c9,2019-04-05 05:42:46.569


In [20]:
total_runs = users_match.groupby('user_id')['home_team_total'].sum()
total_runs = total_runs.to_frame().reset_index()

In [21]:
total_runs.head()

,user_id,home_team_total
0,5ca6ead5b65b15544e169ad6,142.0
1,5ca6ead6d468e03534a810c9,957.0
2,5ca6ec2c93aec35428abbaf9,1387.0
3,5ca6ed54d468e03534a811d2,119.0
4,5ca6eea3800ebb353a469fdb,3186.0


In [22]:
total_runs.columns = ['user_id','total runs']

In [23]:
total_runs.describe()

,total runs
count,12838.000000
mean,1077.210002
std,1440.084075
min,52.000000
25%,142.250000
50%,496.000000
75%,1411.750000
max,14586.000000
